#STURM-Flood

In [1]:
#@title Imports

import random
import os
import sys
import zipfile
import ee
import time

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [2]:
#@title Setup

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
mount_drive = True  #@param {type:"boolean"}
clone_repo = False  #@param {type:"boolean"}
reset_export = False  #@param {type:"boolean"}
gee_export = True  #@param {type:"boolean"}
push_to_HF = False  #@param {type:"boolean"}
cancle_gee_tasks = False  #@param {type:"boolean"}
gee_project = "243624085884"  #@param {type:"string"}

if not mount_drive and not clone_repo:
    raise ValueError("Either mount_drive or clone_repo must be True.")

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

if mount_drive:
    drive.mount(mount_point)
    project_root = os.path.join(root_path, "STURM-fusion") if clone_repo else root_path
else:
    project_root = "STURM-fusion"

if clone_repo and not os.path.exists(project_root):
    !git clone {repo_url} "{project_root}"

project_root = Path(project_root)
assert project_root.exists(), f"Repo not found at {project_root}. Enable clone_repo, mount_drive, or fix root_path."

if gee_export:
    ee.Authenticate(force=True, auth_mode="notebook")
    ee.Initialize(project=gee_project)

sys.path.append(str(project_root))

from config.config import CFG
cfg = CFG()
cfg.ROOT = project_root
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive" if mount_drive else project_root
cfg.GEE_PROJECT = gee_project

print("✅ ROOT:", cfg.ROOT)
print("✅ DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("✅ GEE project:", cfg.GEE_PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=Dwz8eh-5a9EU8wZQ6GPbriGPRE1JVUKYhD3bSzbv82s&tc=kldKq0MBN7CwlhX86mDKQ2s6QElHmJbw6JbvBjaMx2w&cc=VbqTT-IjQYNvqQI_0QImyrwirDkLMJrKvLJmYlposbY

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPyauzSylN4OFHnBzEgNNDRlC-yTJMq_SkMhQFhbEdMNGPXLBG9jkOY

Successfully saved authorization token.
✅ ROOT: /content/drive/MyDrive/MSc/STURM-fu

In [3]:
from src.gee.tasks import cancel_all_tasks
from src.utils.io import clear_export_folder, create_dataset_structure, zip_dataset
from src.data.sturm_flood import download_and_extract
from src.gee.tasks import has_active_tasks
from src.pipeline.build_dataset import process_csv, save_dataframe_to_csv, export_all_s1_images, assemble_dataset, preprocessing_s1_pipeline, preprocessing_s2_pipeline
from src.pipeline.data_validation import validate_files, validate_nan_files, retry_export_of_nan_files, validate_dataset, check_image_shapes, get_band_min_max, get_band_percentiles, get_max_time_difference_with_row
from src.hugging_face.push_dataset import push_zip_to_hf

In [4]:
if cancle_gee_tasks:
  cancel_all_tasks()

In [5]:
if reset_export:
    clear_export_folder(cfg)

create_dataset_structure(cfg)

✅ Dataset structure created:
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-flood
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/S1
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/S2
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/floodmaps
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/metadata


In [6]:
#@title Download STURM-Flood

download_and_extract(cfg)

✅ Zip already exists or dataset present, skipping download.
✅ Dataset already extracted, skipping unzip.


In [ ]:
if gee_export and not has_active_tasks():
    print("No active GEE tasks.starting 🚀"  )
    images, df_fusion = process_csv(cfg.OLD_S2_METADATA_CSV, cfg, verbose=True)
    print('')
    print(len(images), "images processed.")

    save_dataframe_to_csv(df_fusion, cfg.NEW_METADATA_CSV)
    print('New Metadata saved')

    #export_all_s1_images(images, cfg)
    print('S1 images exported')
else:
    print("Active GEE tasks detected. Please wait for them to finish before startingt. ⏳")

Streaming output truncated to the last 5000 lines.
Processing EMSR292_01CHRISOUPOLI_11_05_1_1.tif - S2 image found
s2 time diff: 26.61h - within threshold
Found 2 S1 images for EMSR292_01CHRISOUPOLI_11_05_1_1.tif within time window
Best S1 image for EMSR292_01CHRISOUPOLI_11_05_1_1.tif: S1B_IW_GRDH_1SDV_20180629T043015_20180629T043040_011583_0154B1_CF86 at 29/06/2018 04:30
S1 image for EMSR292_01CHRISOUPOLI_11_05_1_1.tif covers AOI sufficiently
Complete: EMSR292_01CHRISOUPOLI_11_05_1_1.tif


Processing EMSR292_01CHRISOUPOLI_11_05_1_2.tif - S2 image found
s2 time diff: 26.61h - within threshold
Found 2 S1 images for EMSR292_01CHRISOUPOLI_11_05_1_2.tif within time window
Best S1 image for EMSR292_01CHRISOUPOLI_11_05_1_2.tif: S1B_IW_GRDH_1SDV_20180629T043015_20180629T043040_011583_0154B1_CF86 at 29/06/2018 04:30
S1 image for EMSR292_01CHRISOUPOLI_11_05_1_2.tif covers AOI sufficiently
Complete: EMSR292_01CHRISOUPOLI_11_05_1_2.tif


Processing EMSR292_01CHRISOUPOLI_11_05_2_1.tif - S2 image f

In [ ]:
while not validate_files(cfg):
    assemble_dataset(cfg)
    print('Dataset assembled')

    preprocessing_s1_pipeline(cfg)
    print('S1 preprocessing done')

    preprocessing_s2_pipeline(cfg)
    print('S2 preprocessing done')

    time.sleep(120)


📊 Validation Results:
Total rows: 1969
Missing S2: 0
Missing S1: 0
✅ Dataset complete


In [ ]:
if not validate_nan_files(cfg):
    df = retry_export_of_nan_files(cfg)

In [ ]:
if validate_dataset(cfg):

    print('Starting inspection Pipeline')

    result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)
    print(result)

    get_band_min_max(cfg.NEW_S1_PATH)
    get_band_percentiles(cfg.NEW_S1_PATH)

    max_s2_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel2_timestamp")

    print(f"Max S2 time difference: {max_s2_diff['time_diff_hours']:.2f} hours")

    max_s1_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel1_timestamp")

    print(f"Max S1 time difference: {max_s1_diff['time_diff_hours']:.2f} hours")


📊 Validation Results:
Total rows: 1969
Missing S2: 0
Missing S1: 0
✅ Dataset complete


KeyboardInterrupt: 

In [ ]:
if push_to_HF and validate_dataset(cfg):
    zip_dataset(cfg)
    print('Dataset zipped')
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    from huggingface_hub import login

    login(token=HF_TOKEN)

    url = push_zip_to_hf(zip_path=cfg.NEW_ZIP_PATH,repo_id=cfg.HF_REPO_ID,path_in_repo="Dataset.zip",private=False,)
    print(url)


📊 Validation Results:
Total rows: 1969
Missing S2: 0
Missing S1: 0
✅ Dataset complete


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


✅ All files marked as preprocessed
Number of bad files: 0
✅ All files have acceptable NaN ratio
🗜️ Zipping dataset directly...
✅ Dataset zipped at: /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24/Dataset.zip
Dataset zipped


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...URM-fusion-24/Dataset.zip:   0%|          | 6.53MB / 1.38GB            

https://huggingface.co/datasets/tax2310/STURM-fusion-24/commit/5a0e505908e00c9df70fde277e7bfaf2a97fbbca
